In [1]:
import numpy as np
import pandas as pd 

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [2]:
from gensim.models import Word2Vec

In [3]:
sentence1 = "River bank flows"
s1 = sentence1.split(" ")
print(s1)

['River', 'bank', 'flows']


In [4]:
sentence2 = "Money bank grows"
s2 = sentence2.split(" ")
print(s2)

['Money', 'bank', 'grows']


In [5]:
sentences = [s1, s2]
print(sentences)

[['River', 'bank', 'flows'], ['Money', 'bank', 'grows']]


In [6]:
model = Word2Vec(sentences, vector_size=4, window=5, min_count=1, sg=1)

In [7]:
embedding1 = model.wv['bank']
embedding2 = model.wv['Money']

In [8]:
print("Embedding 1 : ", embedding1)

Embedding 1 :  [-0.01340568  0.00591078  0.12758374  0.22523183]


In [9]:
print("Embedding 2 : ", embedding2)

Embedding 2 :  [-0.1253857  -0.09408429  0.18451262 -0.03833678]


In [10]:
dot_product = np.dot(embedding1, embedding2)

In [11]:
print("dot product similarity :", dot_product)

dot product similarity : 0.016030915


In [12]:
#embedding1 = model.wv['lets']
#embedding2 = model.wv['can']
#print("dot product similarity:", np.dot(embedding1, embedding2))

# **self-attention - basic implementation**

1. outputs = contextual embeddings
2. inputs = token/word embeddings
3. processing = weight matrices required

here, embedding dim is 4. 
so, weight matrices must be 4x4.
because (1x4) x (4x4) = (4x4)

aim: apply self attention to each sentence

can be parallelized by using matrices

In [13]:
import torch

In [14]:
# step 1: extract embeddings for all words of the sentence

input_matrix = []
#np_ip = np.array(input_matrix)

for word in s1:
    embedding = model.wv[word]
    input_matrix.append(embedding)

np_ip = np.array(input_matrix)
query_tensors = torch.tensor(np_ip)

print(query_tensors)

tensor([[ 0.0719,  0.0248, -0.2071, -0.2362],
        [-0.0134,  0.0059,  0.1276,  0.2252],
        [-0.1134,  0.1639, -0.1215, -0.0454]])


In [15]:
#query_tensors = torch.tensor([[1.0,2.0,0.0,1.0],[0.0,1.0,-1.0,2.0],[1.0,0.0,-1.0,1.0]])

In [16]:
"""# sample tensor
x = torch.tensor([[1, 2, 3],
                  [4, 5, 6]])

# transpose the tensor, swapping dimensions 0 and 1
y = torch.transpose(x, 0, 1)

print("original tensor: ", x)
print("transposed tensor: ", y)"""

'# sample tensor\nx = torch.tensor([[1, 2, 3],\n                  [4, 5, 6]])\n\n# transpose the tensor, swapping dimensions 0 and 1\ny = torch.transpose(x, 0, 1)\n\nprint("original tensor: ", x)\nprint("transposed tensor: ", y)'

In [17]:
# transpose the input matrix so that it can act as key vector

key_tensors = torch.transpose(query_tensors, 0, 1)
print(key_tensors)

tensor([[ 0.0719, -0.0134, -0.1134],
        [ 0.0248,  0.0059,  0.1639],
        [-0.2071,  0.1276, -0.1215],
        [-0.2362,  0.2252, -0.0454]])


In [18]:
"""mat1 = torch.tensor([[1, 2], [3, 4]])
mat2 = torch.tensor([[5, 6], [7, 8]])
result_matrix = torch.matmul(mat1, mat2)
print(result_matrix)"""

'mat1 = torch.tensor([[1, 2], [3, 4]])\nmat2 = torch.tensor([[5, 6], [7, 8]])\nresult_matrix = torch.matmul(mat1, mat2)\nprint(result_matrix)'

In [19]:
# step 2: perform dot product between query and key tensors
dot_product = torch.matmul(query_tensors, key_tensors)
print(dot_product)

tensor([[ 0.1045, -0.0804,  0.0318],
        [-0.0804,  0.0672, -0.0232],
        [ 0.0318, -0.0232,  0.0565]])


In [20]:
import math

In [21]:
# scale the dot product by 1/sqrt(4) 

sqroot = math.sqrt(4)
scaled_dot_product = (1/sqroot)*dot_product
print(scaled_dot_product)

tensor([[ 0.0522, -0.0402,  0.0159],
        [-0.0402,  0.0336, -0.0116],
        [ 0.0159, -0.0116,  0.0283]])


In [22]:
"""import torch
import torch.nn.functional as F

# Create a sample tensor
x = torch.tensor([[1.0, 2.0, 3.0],
                  [14.0, 5.0, 6.0],
                  [7.0, 8.0, 9.0]], dtype=torch.float)

print("Original tensor:")
print(x)

# Apply softmax to each row (dim=1)
softmax_output = F.softmax(x, dim=1)

print("\nSoftmax output (applied to each row):")
print(softmax_output)

# Verify that each row sums to 1
row_sums = torch.sum(softmax_output, dim=1)
print("\nSum of elements in each row:")
print(row_sums)"""

'import torch\nimport torch.nn.functional as F\n\n# Create a sample tensor\nx = torch.tensor([[1.0, 2.0, 3.0],\n                  [14.0, 5.0, 6.0],\n                  [7.0, 8.0, 9.0]], dtype=torch.float)\n\nprint("Original tensor:")\nprint(x)\n\n# Apply softmax to each row (dim=1)\nsoftmax_output = F.softmax(x, dim=1)\n\nprint("\nSoftmax output (applied to each row):")\nprint(softmax_output)\n\n# Verify that each row sums to 1\nrow_sums = torch.sum(softmax_output, dim=1)\nprint("\nSum of elements in each row:")\nprint(row_sums)'

In [23]:
# step 3: apply softmax to each row of the matrix
weights = torch.softmax(dot_product.T, dim=0).T
print(weights)

tensor([[0.3622, 0.3010, 0.3368],
        [0.3108, 0.3602, 0.3290],
        [0.3365, 0.3185, 0.3450]])


In [24]:
# step 4: multiply it by value matrix (query) to obtain final embeddings

final_embeddings = []

"""for row in weights:
    output_row = torch.zeros(4)
    for w in row:
        for erow in query_tensors:
            output_row += w*erow
    final_embeddings.append(output_row)"""

for row in weights:
    output_row = torch.zeros(4)
    i = 0
    while i<3:
        output_row += row[i]*query_tensors[i]
        i += 1
    final_embeddings.append(output_row)

print(final_embeddings)

[tensor([-0.0162,  0.0659, -0.0775, -0.0330]), tensor([-0.0198,  0.0637, -0.0584, -0.0072]), tensor([-0.0192,  0.0667, -0.0710, -0.0234])]


In [25]:
print(torch.matmul(weights, query_tensors)) # same as the loop. verified.

tensor([[-0.0162,  0.0659, -0.0775, -0.0330],
        [-0.0198,  0.0637, -0.0584, -0.0072],
        [-0.0192,  0.0667, -0.0710, -0.0234]])


# **self-attention using randomly initialized Q,K,V vectors**

In [26]:
# bw 0 and 1
w_query = torch.rand(4,4) # randn for standard normal ditribution
print(w_query)

tensor([[0.1763, 0.5703, 0.6772, 0.2696],
        [0.6040, 0.4662, 0.5609, 0.9235],
        [0.4707, 0.0697, 0.1888, 0.6732],
        [0.8159, 0.7333, 0.5475, 0.2157]])


In [27]:
w_key = torch.rand(4,4) 
print(w_key)

tensor([[0.6170, 0.1955, 0.4083, 0.1710],
        [0.4841, 0.1659, 0.4295, 0.2101],
        [0.6072, 0.3489, 0.9642, 0.7717],
        [0.9220, 0.0371, 0.2255, 0.0043]])


In [28]:
w_value = torch.rand(4,4) 
print(w_value)

tensor([[0.8296, 0.5312, 0.4942, 0.1946],
        [0.8140, 0.5366, 0.3420, 0.9530],
        [0.2685, 0.1715, 0.1688, 0.1847],
        [0.0636, 0.2327, 0.7551, 0.1382]])


In [29]:
queries = torch.matmul(query_tensors, w_query)
print(queries)

tensor([[-0.2626, -0.1351, -0.1058, -0.1481],
        [ 0.2450,  0.1692,  0.1416,  0.1363],
        [-0.0153, -0.0301, -0.0327,  0.0291]])


In [30]:
keys = torch.matmul(query_tensors, w_key)
print(keys)

tensor([[-0.2872, -0.0629, -0.2130, -0.1434],
        [ 0.2797,  0.0512,  0.1709,  0.0984],
        [-0.1063, -0.0391, -0.1033, -0.0789]])


In [31]:
values = torch.matmul(query_tensors, w_value)
print(values)

tensor([[ 0.0092, -0.0390, -0.1693, -0.0333],
        [ 0.0423,  0.0703,  0.1870,  0.0577],
        [ 0.0038, -0.0037, -0.0548,  0.1054]])


In [32]:
keys = torch.transpose(keys, 0, 1)
print(keys)

tensor([[-0.2872,  0.2797, -0.1063],
        [-0.0629,  0.0512, -0.0391],
        [-0.2130,  0.1709, -0.1033],
        [-0.1434,  0.0984, -0.0789]])


In [33]:
# find dot product
dot_product2 = torch.matmul(queries, keys)
print(dot_product2)

tensor([[ 0.1277, -0.1130,  0.0558],
        [-0.1307,  0.1148, -0.0580],
        [ 0.0091, -0.0085,  0.0039]])


In [34]:
# scale the dot product by 1/sqrt(4) 
sqroot2 = math.sqrt(4)
scaled_dot_product2 = (1/sqroot2)*dot_product2
print(scaled_dot_product2)

tensor([[ 0.0638, -0.0565,  0.0279],
        [-0.0654,  0.0574, -0.0290],
        [ 0.0045, -0.0043,  0.0019]])


In [35]:
# step 3: apply softmax to each row of the matrix
weights2 = torch.softmax(dot_product2.T, dim=0).T
print(weights2)

tensor([[0.3681, 0.2893, 0.3426],
        [0.2982, 0.3812, 0.3207],
        [0.3359, 0.3300, 0.3341]])


In [36]:
final_embeddings2 = []

for row in weights2:
    output_row = torch.zeros(4)
    i = 0
    while i<3:
        output_row += row[i]*values[i]
        i += 1
    final_embeddings2.append(output_row)

print(final_embeddings2)

[tensor([ 0.0169,  0.0047, -0.0270,  0.0405]), tensor([0.0201, 0.0140, 0.0032, 0.0459]), tensor([ 0.0183,  0.0089, -0.0135,  0.0431])]
